In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
#=============== 자료 읽기 ===============
# sst, precip, uwnd, vwnd 네 개를 xr.open_dataset(...)['변수'] 로 읽어오기 
# print(uwnd), print(uwnd['level']) 등으로 데이터 확인 
sst = xr.open_dataset('../data/sst.mnmean.nc')['sst']
precip = xr.open_dataset('../data/precip.mon.mean.nc')['precip']
uwnd = xr.open_dataset('../data/uwnd.mon.mean.nc')['uwnd']
vwnd = xr.open_dataset('../data/vwnd.mon.mean.nc')['vwnd']



In [ ]:
uvwnd = xr.merge([uwnd[:,:,::3,::3], vwnd[:,:,::3,::3]]) # u, v wind 각각을 전체 타임 , 전체 레벨, 3x3 격자 간격으로 추출 후 합친다 

In [ ]:
uvwnd.sel(time='YYYY-MM-DD', level=850) # 850hpa의 자료 가져오기 time은 각자 선택 

In [ ]:
#=============== HW1 ===============  (한 셀)
# PDF에 있는 그림 재현 
fig = plt.figure(figsize=(18, 14), dpi=100)
clevs = np.arange(-3, 33, 1) # 색은 SST로 그릴 것이므로 SST의 범위로 지정 
ax = plt.axes(projection=ccrs.Robinson(central_longitude=180)) # 중심 경도 180 → 태평양이 가운데 겹쳐그리려면 같은 ax를 사용하는 편이 좋다. 
# 그리는 순서 = 겹치는 순서 : 색 → 등고선 → 화살표 등... 

# --- 1) SST   : contourf ---

# sst.sel

# --- 2) 강수  : contour  ---

# precip.sel

# --- 3) 바람  : quiver   --- 
uvwnd.sel(time='2020-08-01', level=850).plot.quiver(ax=ax, x='lon', y='lat', u='uwnd', v='vwnd', 
                                                  color='green', transform=ccrs.PlateCarree())


ax.set_title('SST, Precipitation, Wind_850hPa')


plt.show()



In [ ]:
#=============== HW2 ===============
#climatological 850hPa wind vector (DJF, MAM, JJA, SON) overlaid on climatological precipitation
#-------- 기후값 계산 ----------
# groupby('time.season') — 'time.month'와 같은 방식, 결과는 season 차원 (DJF, JJA, MAM, SON)

precip = xr.open_dataset('../data/precip.mon.mean.nc')['precip']
uwnd = xr.open_dataset('../data/uwnd.mon.mean.nc')['uwnd']
vwnd = xr.open_dataset('../data/vwnd.mon.mean.nc')['vwnd']
uvwnd = xr.merge([uwnd[:,:,::3,::3], vwnd[:,:,::3,::3]]) # u, v wind 각각을 전체 타임 , 전체 레벨, 3x3 격자 간격으로 추출 후 합친다

# precip_clim
# uvwnd_clim 


In [ ]:

#-------- Plotting: 4패널 ----------
# subplot(2, 2, n) : 2행 2열 중 n번 칸 (1부터 셈)
#   221 222
#   223 224
# 강수 기후값(contourf) 위에 850 hPa 바람 기후값(quiver), 계절별로 한 칸씩


#------------- Plotting a Map---------------
# fig 
# clevel 

# Sub-Figure 1 DJF
ax1 = plt.subplot(221, projection=ccrs.Robinson(central_longitude=180)) 


# Sub-Figure 2 MAM
ax2 = plt.subplot(222, projection=ccrs.Robinson(central_longitude=180))

# Sub-Figure 3 JJA
ax3 = plt.subplot(223, projection=ccrs.Robinson(central_longitude=180))


# Sub-Figure 4 SON
ax4 = plt.subplot(224, projection=ccrs.Robinson(central_longitude=180))


plt.show()


# HW2 해석 (3~5줄) →

In [ ]:

#=============== HW3 ===============
# 슬라이드의 코드를 그대로 붙여넣고, 각 줄 끝에 "# ←" 그 줄이 하는 일을 자기 말로

import matplotlib.pyplot as plt
import xarray as xr

dset = xr.open_dataset('../data/sst.mnmean.nc')
sst=dset.sst.sel()

nino34 = sst.where((sst.lat<5) & (sst.lat>-5) & (sst.lon>190) & (sst.lon<240), drop=True).mean(dim=['lat','lon'])

clim = nino34.sel(time=slice('1982-01','2018-12')).groupby('time.month').mean(dim='time')   

anom = (nino34.groupby('time.month') - clim).sel(time=slice('1981-12','2019-02'))

nino3mon = anom.rolling(time=5, center=True).mean()

ninos = nino3mon[12::12]   

fig = plt.figure(figsize = (8,6), dpi = 150) 
ax = fig.subplots()

ax.plot(ninos.time.values, ninos, 'b-') 

ax.axhline(0,color='black',linewidth=0.5)
ax.axhline(1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(-1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(-1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(2.0,color='black',linewidth=0.7,linestyle='dashed')
ax.axhline(-2.0,color='black',linewidth=0.7,linestyle='dashed')
ax.axhline(ninos.std().values,color='red',linewidth=0.5,linestyle='dashed')
ax.set_xlabel('Time')
ax.set_ylabel('[$\degree$C]')
ax.set_title('Nino 3.4 index')

plt.savefig('nino34_index_season.jpg')
plt.show()

